## Scenario 3: Multiple data scientists working on multiple ML models

MLflow setup:
* Tracking server: yes, remote server (EC2).
* Backend store: postgresql database.
* Artifacts store: s3 bucket.

The experiments can be explored by accessing the remote server.

The example uses AWS to host a remote server. In order to run the example you'll need an AWS account. Follow the steps described in the file `mlflow_on_aws.md` to create a new AWS account and launch the tracking server. 

In [ ]:
import mlflow
import os

os.environ["AWS_ACCESS_KEY_ID"] = ""
os.environ["AWS_SECRET_ACCESS_KEY"] = ""
os.environ["AWS_DEFAULT_REGION"] = ""


TRACKING_SERVER_HOST = "ec2-3-226-251-128.compute-1.amazonaws.com"
mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:5000")

In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'http://ec2-3-226-251-128.compute-1.amazonaws.com:5000'


La URI es el DNS de la instancia EC2 creada en AWS

In [3]:
mlflow.search_experiments() # list_experiments API has been removed, you can use search_experiments instead.()

[<Experiment: artifact_location='s3://mlflow-scenario-3-artifacts/0', creation_time=1762875978139, experiment_id='0', last_update_time=1762875978139, lifecycle_stage='active', name='Default', tags={}>]

Aquí se puede ver cómo los artifacts se guardan en el bucket de S3 que se ha creado

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

mlflow.set_experiment("my-experiment-1")

with mlflow.start_run():

    X, y = load_iris(return_X_y=True)

    params = {"C": 0.1, "random_state": 42}
    mlflow.log_params(params)

    lr = LogisticRegression(**params).fit(X, y)
    y_pred = lr.predict(X)
    mlflow.log_metric("accuracy", accuracy_score(y, y_pred))

    mlflow.sklearn.log_model(lr, name="models")
    print(f"default artifacts URI: '{mlflow.get_artifact_uri()}'")

2025/11/11 15:58:08 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
/home/codespace/anaconda3/envs/exp-tracking-env/lib/python3.9/site-packages/boto3/compat.py:84: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


default artifacts URI: 's3://mlflow-scenario-3-artifacts/1/79e2d12aa3e64f7abecdaff42bb9ddf2/artifacts'
🏃 View run indecisive-midge-666 at: http://ec2-3-226-251-128.compute-1.amazonaws.com:5000/#/experiments/1/runs/79e2d12aa3e64f7abecdaff42bb9ddf2
🧪 View experiment at: http://ec2-3-226-251-128.compute-1.amazonaws.com:5000/#/experiments/1


In [3]:
mlflow.search_experiments()

[<Experiment: artifact_location='s3://mlflow-scenario-3-artifacts/1', creation_time=1762876258320, experiment_id='1', last_update_time=1762876258320, lifecycle_stage='active', name='my-experiment-1', tags={}>,
 <Experiment: artifact_location='s3://mlflow-scenario-3-artifacts/0', creation_time=1762875978139, experiment_id='0', last_update_time=1762875978139, lifecycle_stage='active', name='Default', tags={}>]

### Interacting with the model registry

In [4]:
from mlflow.tracking import MlflowClient


client = MlflowClient(f"http://{TRACKING_SERVER_HOST}:5000")

In [5]:
client.search_registered_models()

[]

In [6]:
run_id = client.search_runs(experiment_ids=['1'])[0].info.run_id
mlflow.register_model(
    model_uri=f"runs:/{run_id}/models",
    name='iris-classifier'
)

Successfully registered model 'iris-classifier'.
2025/11/11 15:59:48 WARNING mlflow.tracking._model_registry.fluent: Run with id 79e2d12aa3e64f7abecdaff42bb9ddf2 has no artifacts at artifact path 'models', registering model based on models:/m-d79a5efcb82143498a82b3e59d75c991 instead
2025/11/11 15:59:49 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: iris-classifier, version 1
Created version '1' of model 'iris-classifier'.


<ModelVersion: aliases=[], creation_timestamp=1762876789160, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1762876789160, metrics=None, model_id=None, name='iris-classifier', params=None, run_id='79e2d12aa3e64f7abecdaff42bb9ddf2', run_link='', source='models:/m-d79a5efcb82143498a82b3e59d75c991', status='READY', status_message=None, tags={}, user_id='', version='1'>